In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import torch
from types import SimpleNamespace
from torch.utils.data import TensorDataset, random_split
from train import run_training
import pandas as pd

from utils_functions.utils import (
    decoded_x_to_nas201_arch,
    query_nas201_accuracy,
    set_seed,
)
from datasets.dataset_loader_nas201 import NASDatasetFactory,load_nas201_api
from utils_functions.utilsnas301 import (decode_population_nas301)
from utils_functions.tests_utils import run_flow_vs_random_experiment


tensorflow is not installed.


# NAS 201 tests

In [2]:
DATASET_NAME = "NAS201"
api = load_nas201_api()

train_dataset,test_dataset,train_loader,test_loader = NASDatasetFactory.create(
    benchmark_name="NAS201",
    api=api,
    dataset_name="cifar10",     
    metric="test-accuracy",
    flatten=True,
    normalize_y=True,
)
print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


Train size: 12500
Test size: 3125


In [3]:
SEED = 42
DEVICE = "cuda"

args = SimpleNamespace(
    outer_epochs=1,
    N=256,
    latent_dim=16,
    batch_size=64,

    vae_epochs=200,
    pretrain_vae_epochs=300,
    pretrain_batch_size=64,
    pretrain_fraction=0.2,

    flow_epochs=200,
    alpha=0.5,

    beta=0.0,
    lambda_acc=1.0,

    use_top_mutations=False,
    elite_fraction=0.1,
    mutation_fraction=0.2,
    mutation_k=1,

    benchmark_name="NAS201",
    dataset_name="cifar10",

    nas_hp="200",
    nas_metric="test-accuracy",

    train_dataset=train_dataset,
    test_dataset=test_dataset,

    min_delta=0.01,
    weight_sharing=False,

    seed=SEED,
    device=DEVICE,
)

In [5]:
from utils_functions.tests_utils import run_flow_vs_random_experiment

datasets = [
    "cifar10",
    "cifar100",
    "ImageNet16-120",
]

all_results = {}

for dataset_name in datasets:
    args.dataset_name = dataset_name

    results_df, samples_df, summary_df = run_flow_vs_random_experiment(
        base_args=args,
        run_training_fn=run_training,
        n_runs=1,
        max_test_samples=None,
        initial_seed=42,
        output_dir=f"results_flow_vs_random/{dataset_name}",
    )

    all_results[dataset_name] = {
        "results_df": results_df,
        "samples_df": samples_df,
        "summary_df": summary_df,
    }


RUN 1/1 | NAS201 | cifar10 | seed=42
Dataset available

PRETRAIN VAE
VAE pretrain epoch 000 | loss=1.220420 | recon=0.952654 | kl=0.097304 | acc_loss=0.267766
VAE pretrain epoch 050 | loss=0.004024 | recon=0.000340 | kl=48.744926 | acc_loss=0.003685
VAE pretrain epoch 100 | loss=0.000882 | recon=0.000063 | kl=58.561596 | acc_loss=0.000820
VAE pretrain epoch 150 | loss=0.000483 | recon=0.000036 | kl=45.833284 | acc_loss=0.000447
Early stopping: patience reached at epoch 197, best_loss=0.000364
VAE pretrained and frozen.

 OUTER EPOCH 1/1 ==========

Evaluated NAS201 architectures:
valid archs = 256 / 256
mean acc    = 0.8690
std acc     = 0.1206
min acc     = 0.1000
max acc     = 0.9406
z_all shape: torch.Size([256, 16])
y_all shape: torch.Size([256])
Number of pairs: 226
pairs_x shape: torch.Size([226, 16])
pairs_target shape: torch.Size([226, 16])
z_new shape: torch.Size([256, 16])
Campioni validi:          3125
Baseline media:           0.873392
Miglioramento Flow:       -0.000624
M

In [13]:
comparison_rows = []

display_names = {
    "cifar10": "CIFAR-10",
    "cifar100": "CIFAR-100",
    "ImageNet16-120": "ImageNet16",
}

for dataset_name, result in all_results.items():
    results_df = result["results_df"]

    start_acc = results_df["mean_start_accuracy"].mean()
    delta_f = results_df["mean_flow_improvement"].mean()
    delta_r = results_df["mean_random_improvement"].mean()
    advantage = results_df["mean_flow_advantage"].mean()

    comparison_rows.append({
        "Dataset": display_names[dataset_name],
        "start acc": f"{start_acc * 100:.2f}%",
        "∆F": f"{delta_f * 100:+.2f}%",
        "∆R": f"{delta_r * 100:+.2f}%",
        "∆F − ∆R": f"{advantage * 100:+.2f}%",
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

,Dataset,start acc,∆F,∆R,∆F − ∆R
0,CIFAR-10,87.34%,-0.06%,-0.33%,+0.27%
1,CIFAR-100,61.67%,+0.15%,-0.23%,+0.39%
2,ImageNet16,33.75%,+0.76%,-0.14%,+0.91%


## with weight sharing:

In [17]:

args = SimpleNamespace(
    benchmark_name="NAS201",

    n_samples=1000,
    performance_model=None,

    outer_epochs=1,
    N=20,
    latent_dim=32,
    batch_size=64,

    vae_epochs=5,
    pretrain_vae_epochs=200,
    pretrain_batch_size=64,
    pretrain_fraction=0.5,

    flow_epochs=5,
    alpha=0.5,

    beta=0.0,
    lambda_acc=1.0,

    use_top_mutations=False,
    elite_fraction=0.2,
    mutation_fraction=0.2,
    mutation_k=1,

    benchmark_mane="nas301",
    dataset_name="cifar10",
    nas_hp=None,
    nas_metric="surrogate_accuracy",

    train_dataset=None,
    test_dataset=None,
    weight_sharing = True,

    pos_weight_value=5,
    min_delta = 0.001,
    seed=42,
    device="cuda",
)

In [ ]:
# from utils_functions.tests_utils import run_flow_vs_random_experiment

# results_df, samples_df, summary_df = run_flow_vs_random_experiment(
#     base_args=args,
#     run_training_fn=run_training,
#     n_runs=1,
#     max_test_samples=None,
#     initial_seed=42,
#     output_dir="results_flow_vs_random"
)


RUN 1/1 | NAS201 | cifar10 | seed=42

PRETRAIN VAE
VAE pretrain epoch 000 | loss=1.350278 | recon=1.350278 | kl=0.007190 | acc_loss=0.000000
VAE pretrain epoch 050 | loss=0.003243 | recon=0.003243 | kl=24.727674 | acc_loss=0.000000
VAE pretrain epoch 100 | loss=0.000144 | recon=0.000144 | kl=36.149041 | acc_loss=0.000000
Early stopping: loss below threshold at epoch 113, loss=0.000100
VAE pretrained and frozen.

 OUTER EPOCH 1/1 ==========
Evaluating networks...
Epoch   0 | Batch    0 | Loss: 2.4710 | LR: 0.025000 | Path noti: 20
Epoch   0 | Batch   25 | Loss: 2.2485 | LR: 0.025000 | Path noti: 20
Epoch   0 | Batch   50 | Loss: 2.2346 | LR: 0.025000 | Path noti: 20
Epoch   0 | Batch   75 | Loss: 2.2283 | LR: 0.025000 | Path noti: 20
Epoch   0 | Batch  100 | Loss: 2.1669 | LR: 0.025000 | Path noti: 20
Epoch   0 | Batch  125 | Loss: 2.1694 | LR: 0.025000 | Path noti: 20
Epoch   0 | Batch  150 | Loss: 2.1635 | LR: 0.025000 | Path noti: 20
Epoch   0 | Batch  175 | Loss: 2.1328 | LR: 0.025

UnboundLocalError: cannot access local variable 'test_dataset' where it is not associated with a value

In [ ]:
comparison_rows = []

display_names = {
    "cifar10": "CIFAR-10",
    "cifar100": "CIFAR-100",
    "imagenet16-120": "ImageNet16",
}

for dataset_name, result in all_results.items():
    results_df = result["results_df"]

    start_acc = results_df["mean_start_accuracy"].mean()
    delta_f = results_df["mean_flow_improvement"].mean()
    delta_r = results_df["mean_random_improvement"].mean()
    advantage = results_df["mean_flow_advantage"].mean()

    comparison_rows.append({
        "Dataset": display_names[dataset_name],
        "start acc": f"{start_acc * 100:.2f}%",
        "∆F": f"{delta_f * 100:+.2f}%",
        "∆R": f"{delta_r * 100:+.2f}%",
        "∆F − ∆R": f"{advantage * 100:+.2f}%",
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

# NAS 301 tests

In [2]:

def load_csv_as_dataset(csv_path):
    df = pd.read_csv(csv_path)
    feature_cols = [col for col in df.columns if col.startswith("x_")]

    feature_cols = sorted(
        feature_cols,
        key=lambda c: int(c.split("_")[1])
    )

    X = df[feature_cols].values
    Y = df["accuracy"].values

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)

    dataset = TensorDataset(X, Y)

    print("CSV caricato:", csv_path)
    print("Numero esempi:", len(dataset))
    return X, Y, dataset

X, Y, dataset_301 = load_csv_as_dataset(
    "../datasets/nas301/nas301_dataset.csv")

CSV caricato: ../datasets/nas301/nas301_dataset.csv
Numero esempi: 50000


In [3]:
train_size = int(0.8 * len(dataset_301))
test_size = len(dataset_301) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, test_dataset = random_split(
    dataset_301,
    [train_size, test_size],
    generator=generator
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 40000
Test size: 10000


In [5]:
args = SimpleNamespace(
    benchmark_name="NAS301",
    n_samples=20,          # piccolo per test veloce
    performance_model=None,  # lo carica la factory se return_model=True
    outer_epochs=1,
    N=20,
    latent_dim=32,
    batch_size=64,

    vae_epochs=5,
    pretrain_vae_epochs=200,
    pretrain_batch_size=64,
    pretrain_fraction=0.5,

    flow_epochs=5,
    alpha=0.5,

    beta=0.0,
    lambda_acc=1.0,

    use_top_mutations=False,
    elite_fraction=0.2,
    mutation_fraction=0.2,
    mutation_k=1,

    benchmark_mane = "nas301",
    dataset_name="cifar10",
    nas_hp=None,
    nas_metric="surrogate_accuracy",

    train_dataset=train_dataset,
    test_dataset=test_dataset,
    pos_weight_value = 5,
    weight_sharing = False,
    min_delta = 0.001,

    seed=42,
    device="cuda",
)

history, model_VAE, flow, test_dataset, api_or_model = run_training(args)

Pesi NAS-Bench-301 trovati in c:\Users\aless\Desktop\Deep Lerning\progetto\flownas\Deep-Learning-Project\datasets\nb_models_1.0\xgb_v1.0
Surrogate NAS-Bench-301 caricato.
Dataset available

PRETRAIN VAE
VAE pretrain epoch 000 | loss=0.334534 | recon=0.322161 | kl=1.134340 | acc_loss=0.012373
VAE pretrain epoch 050 | loss=0.100716 | recon=0.100666 | kl=9.231310 | acc_loss=0.000050
VAE pretrain epoch 100 | loss=0.049489 | recon=0.049431 | kl=11.383883 | acc_loss=0.000058
VAE pretrain epoch 150 | loss=0.030185 | recon=0.030149 | kl=12.274001 | acc_loss=0.000035
VAE pretrained and frozen.

 OUTER EPOCH 1/1 ==========

Evaluated NAS301 architectures:
valid archs = 20 / 20
mean acc    = 0.9310
std acc     = 0.0056
min acc     = 0.9198
max acc     = 0.9383
z_all shape: torch.Size([20, 32])
y_all shape: torch.Size([20])
Number of pairs: 6
pairs_x shape: torch.Size([6, 32])
pairs_target shape: torch.Size([6, 32])
z_new shape: torch.Size([20, 32])


In [6]:
results_df, samples_df, summary_df = run_flow_vs_random_experiment(
    base_args=args,
    run_training_fn=run_training,
    n_runs=1,
    max_test_samples=None,
    initial_seed=42,
    output_dir="results_flow_vs_random"
)

start_acc = results_df["mean_start_accuracy"].mean()
delta_f = results_df["mean_flow_improvement"].mean()
delta_r = results_df["mean_random_improvement"].mean()

comparison_df = pd.DataFrame([{
    "Dataset": "CIFAR-10",
    "start acc": f"{start_acc * 100:.2f}%",
    "∆F": f"{delta_f * 100:+.2f}%",
    "∆R": f"{delta_r * 100:+.2f}%",
    "∆F − ∆R": f"{(delta_f - delta_r) * 100:+.2f}%",
}])

comparison_df


RUN 1/1 | NAS301 | cifar10 | seed=42
Pesi NAS-Bench-301 trovati in c:\Users\aless\Desktop\Deep Lerning\progetto\flownas\Deep-Learning-Project\datasets\nb_models_1.0\xgb_v1.0
Surrogate NAS-Bench-301 caricato.
Dataset available

PRETRAIN VAE
VAE pretrain epoch 000 | loss=0.334534 | recon=0.322161 | kl=1.134340 | acc_loss=0.012373
VAE pretrain epoch 050 | loss=0.100716 | recon=0.100666 | kl=9.231310 | acc_loss=0.000050
VAE pretrain epoch 100 | loss=0.049489 | recon=0.049431 | kl=11.383883 | acc_loss=0.000058
VAE pretrain epoch 150 | loss=0.030185 | recon=0.030149 | kl=12.274001 | acc_loss=0.000035
VAE pretrained and frozen.

 OUTER EPOCH 1/1 ==========

Evaluated NAS301 architectures:
valid archs = 20 / 20
mean acc    = 0.9310
std acc     = 0.0056
min acc     = 0.9198
max acc     = 0.9383
z_all shape: torch.Size([20, 32])
y_all shape: torch.Size([20])
Number of pairs: 6
pairs_x shape: torch.Size([6, 32])
pairs_target shape: torch.Size([6, 32])
z_new shape: torch.Size([20, 32])
Campioni v

,Dataset,start acc,∆F,∆R,∆F − ∆R
0,CIFAR-10,93.07%,+0.11%,+0.01%,+0.10%
